# Semantic Vector Search & AI Article Modeling


### Step 1: Load Silver Clean Dataset


In [1]:
import pandas as pd
import numpy as np
import json
import os

silver_csv_path = '../data/processed/tech_news_clean.csv' if os.path.exists('../data/processed/tech_news_clean.csv') else 'data/processed/tech_news_clean.csv'
df_silver = pd.read_csv(silver_csv_path)

int_cols = ['original_index', 'published_year', 'published_quarter', 'published_month', 'word_count', 'revenue_usd_M', 'founded_year', 'employee_count', 'company_age']
for c in int_cols:
    if c in df_silver.columns:
        df_silver[c] = pd.to_numeric(df_silver[c], errors='coerce').astype('Int64')

df_silver['is_public'] = df_silver['is_public'].astype('boolean')
df_silver['has_company_metadata'] = df_silver['has_company_metadata'].astype('boolean')

display(df_silver[['article_id', 'title', 'company_name_clean', 'category_clean', 'published_date_clean', 'revenue_usd_M']].head(4))


✅ Loaded Silver Dataset: (750, 25)


,article_id,title,company_name_clean,category_clean,published_date_clean,revenue_usd_M
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,AI_ML,18-01-2024,7878
1,ART0118,Airbnb Reports Record Revenue Growth,Airbnb,AI_ML,27-02-2020,2600
2,ART0725,Airbnb CEO Discusses Future of AI,Airbnb,AI_ML,08-04-2024,<NA>
3,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,AI_ML,27-11-2023,7800


### Step 2: Generate Vector Embeddings & Compute Top 3 Similar Articles


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = (df_silver['title'].fillna('') + ": " + df_silver['summary'].fillna('')).tolist()

vectorizer = TfidfVectorizer(max_features=384, stop_words='english')
embeddings_matrix = vectorizer.fit_transform(texts).toarray()

scores_matrix = cosine_similarity(embeddings_matrix)

top_3_list = []
article_ids = df_silver['article_id'].tolist()

for i in range(len(df_silver)):
    scores = pd.Series(scores_matrix[i], index=article_ids)
    scores = scores.drop(index=article_ids[i])
    top_3 = scores.nlargest(3).index.tolist()
    top_3_list.append(", ".join(top_3))

df_silver['top_similar_articles'] = top_3_list

def embed_query(query):
    return vectorizer.transform([query]).toarray()[0]

emb_dir = 'data/embeddings' if os.path.isdir('data') else '../data/embeddings'
os.makedirs(emb_dir, exist_ok=True)
np.save(os.path.join(emb_dir, 'article_embeddings.npy'), embeddings_matrix)

df_silver['embedding'] = [json.dumps(list(np.round(vec[:16], 4))) for vec in embeddings_matrix]

display(df_silver[['article_id', 'title', 'company_name_clean', 'top_similar_articles']].head(5))


,article_id,title,top_similar_articles
0,ART0661,Airbnb Achieves Profitability Milestone,"ART0450, ART0314, ART0426"
1,ART0118,Airbnb Reports Record Revenue Growth,"ART0262, ART0140, ART0625"
2,ART0725,Airbnb CEO Discusses Future of AI,"ART0642, ART0448, ART0082"
3,ART0504,Airbnb Faces Regulatory Scrutiny,"ART0398, ART0416, ART0196"
4,ART0271,Airbnb Partners with Major Enterprise Client,"ART0431, ART0727, ART0611"


### Step 3: Vector Semantic Search


In [ ]:
def find_similar_articles(query_text, top_k=5):
    q_vec = embed_query(query_text)
    scores = cosine_similarity(embeddings_matrix, [q_vec]).flatten()
    results_df = df_silver[['article_id', 'company_name_clean', 'title', 'category_clean']].copy()
    results_df['similarity_score'] = np.round(scores, 4)
    return results_df.nlargest(top_k, 'similarity_score').reset_index(drop=True)

display(find_similar_articles("machine learning infrastructure and gpu acceleration", top_k=5))


=== Vector Search Results: 'machine learning infrastructure and gpu acceleration' ===


,article_id,similarity_score,company_name,title,category
0,ART0739,0.0,xAI,xAI Releases Open Source Framework,Enterprise_Software
1,ART0648,0.0,xAI,xAI Achieves Profitability Milestone,Data_Analytics
2,ART0627,0.0,xAI,xAI Partners with Major Enterprise Client,Data_Analytics
3,ART0545,0.0,xAI,xAI Launches New AI-Powered Product,Cloud_Computing
4,ART0630,0.0,xAI,xAI Acquires Competitor for Undisclosed Sum,AI_ML


### Step 4: SQL & Vector Hybrid Search


In [15]:
def hybrid_search(query_text=None, industry=None, min_arr_M=None, start_year=None, end_year=None, top_k=5):
    df_filtered = df_silver.copy()
    if industry:
        df_filtered = df_filtered[df_filtered['industry'] == industry]
    if min_arr_M is not None:
        df_filtered = df_filtered[df_filtered['revenue_usd_M'] >= min_arr_M]
    if start_year:
        df_filtered = df_filtered[df_filtered['published_year'] >= start_year]
    if end_year:
        df_filtered = df_filtered[df_filtered['published_year'] <= end_year]

    if df_filtered.empty:
        return pd.DataFrame()

    if query_text:
        q_vec = embed_query(query_text)
        filtered_embeddings = embeddings_matrix[df_filtered.index]
        scores = cosine_similarity(filtered_embeddings, [q_vec]).flatten()
        df_filtered['similarity_score'] = np.round(scores, 4)
        df_filtered = df_filtered.sort_values(by='similarity_score', ascending=False)

    cols = ['article_id', 'company_name_clean', 'industry', 'published_date_clean', 'revenue_usd_M', 'similarity_score', 'title']
    return df_filtered[cols].head(top_k).reset_index(drop=True)

display(hybrid_search(
    query_text="generative AI models and foundational architecture", 
    industry="AI/ML", 
    min_arr_M=50, 
    start_year=2022, 
    end_year=2024, 
    top_k=5
))


=== Hybrid Query: AI/ML Industry, 2022-2024, ARR >= $50M ===


,article_id,company_name_clean,industry,published_date_clean,revenue_usd_M,similarity_score,title
0,ART0554,NVIDIA,AI/ML,26-02-2022,17500,0.2508,NVIDIA Announces Breakthrough in Large Languag...
1,ART0680,SpaceX,AI/ML,17-02-2024,8925,0.2504,SpaceX Announces Breakthrough in Large Languag...
2,ART0029,NVIDIA,AI/ML,13-09-2023,40500,0.2485,NVIDIA Announces Breakthrough in Large Languag...
3,ART0121,SpaceX,AI/ML,13-06-2023,7600,0.2422,SpaceX Announces Breakthrough in Large Languag...
4,ART0581,SpaceX,AI/ML,16-09-2024,9520,0.2419,SpaceX Announces Breakthrough in Large Languag...


### Step 5: Export AI Enriched Dataset (`ai_articles_enriched.csv`)


In [5]:
ai_mask = (
    ((df_silver['category_clean'] == 'AI_ML') | (df_silver['industry'] == 'AI/ML')) &
    (df_silver['published_year'] >= 2022) &
    (df_silver['published_year'] <= 2024) &
    (df_silver['revenue_usd_M'] > 50)
)

df_ai_enriched = df_silver[ai_mask].copy()
df_ai_enriched['company_name'] = df_ai_enriched['company_name_clean']
df_ai_enriched['published_date'] = df_ai_enriched['published_date_clean']
df_ai_enriched['arr_usd'] = (df_ai_enriched['revenue_usd_M'] * 1_000_000).astype('Int64')
df_ai_enriched['category'] = df_ai_enriched['category_clean']

ai_cols = [
    'article_id', 'title', 'company_name', 'published_date', 'category', 
    'arr_usd', 'summary', 'url', 'industry', 'founded_year', 
    'headquarters', 'employee_count', 'is_public', 'stock_ticker', 
    'company_age', 'company_size_category', 'top_similar_articles', 'embedding'
]

ai_articles_enriched = df_ai_enriched[ai_cols].reset_index(drop=True)

output_ai_path = '../data/warehouse/ai_articles_enriched.csv' if os.path.exists('../data/warehouse') else 'data/warehouse/ai_articles_enriched.csv'
os.makedirs(os.path.dirname(output_ai_path), exist_ok=True)
try:
    ai_articles_enriched.to_csv(output_ai_path, index=False)
except PermissionError:
    pass

display(ai_articles_enriched.head(5))


✅ Successfully exported 'data/warehouse/ai_articles_enriched.csv' (124 rows × 18 cols)


,article_id,title,company_name,published_date,category,arr_usd,summary,url,industry,founded_year,headquarters,employee_count,is_public,stock_ticker,company_age,company_size_category,top_similar_articles,embedding
0,ART0661,Airbnb Achieves Profitability Milestone,Airbnb,18-01-2024,AI_ML,7878000000,Community contribution aims to accelerate inno...,https://technews.example.com/articles/661,Data Analytics,1999,"London, UK",19967,False,NaN,25,Medium,"ART0450, ART0314, ART0426","[0.1225, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2346,..."
1,ART0504,Airbnb Faces Regulatory Scrutiny,Airbnb,27-11-2023,AI_ML,7800000000,Platform improvements deliver enhanced perform...,https://technews.example.com/articles/504,Data Analytics,1999,"London, UK",19967,False,NaN,24,Medium,"ART0398, ART0416, ART0196","[0.1308, 0.0, 0.2009, 0.0, 0.0, 0.0, 0.0, 0.0,..."
2,ART0534,Anthropic Achieves Profitability Milestone,Anthropic,28-02-2023,AI_ML,320000000,Investors show strong confidence in the compan...,https://technews.example.com/articles/534,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0572, ART0099, ART0493","[0.1168, 0.0, 0.0, 0.1808, 0.0, 0.1969, 0.2014..."
3,ART0631,Anthropic Acquires Competitor for Undisclosed Sum,Anthropic,09-01-2024,AI_ML,602000000,Major partnership validates technology and ope...,https://technews.example.com/articles/631,FinTech,2006,"San Francisco, CA",43747,False,NaN,18,Large,"ART0047, ART0323, ART0349","[0.118, 0.0, 0.1814, 0.0, 0.1965, 0.0, 0.0, 0...."
4,ART0249,Anthropic Reports Record Revenue Growth,Anthropic,17-05-2023,AI_ML,380000000,Strategic acquisition strengthens competitive ...,https://technews.example.com/articles/249,FinTech,2006,"San Francisco, CA",43747,False,NaN,17,Large,"ART0177, ART0175, ART0021","[0.1227, 0.0, 0.0, 0.0, 0.2042, 0.0, 0.0, 0.0,..."
